In [3]:
import customtkinter as ctk
from tkinter import filedialog, messagebox
import cv2
from PIL import Image
import os

# 영상/음성 분리를 위한 라이브러리
try:
    from moviepy import VideoFileClip
except ImportError:
    VideoFileClip = None

# CustomTkinter 기본 테마 설정
ctk.set_appearance_mode("System")  
ctk.set_default_color_theme("blue")

class MockInterviewApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title("AI 모의면접 솔루션 (CustomTkinter)")
        self.geometry("1000x650")

        # 상태 변수
        self.vid = None
        self.video_writer = None
        self.is_recording = False
        self.current_video_path = ""
        self.delay = 15 # 영상 프레임 업데이트 딜레이 (ms)
        self.loop_id = None # 프레임 반복 루프를 제어할 ID (새로 추가)

        # --- UI 레이아웃 구성 ---
        self.grid_columnconfigure(0, weight=1)
        self.grid_columnconfigure(1, weight=0)
        self.grid_rowconfigure(0, weight=1)

        # 1. 왼쪽: 영상 출력부 프레임
        self.video_frame = ctk.CTkFrame(self, corner_radius=15)
        self.video_frame.grid(row=0, column=0, padx=20, pady=20, sticky="nsew")
        self.video_frame.grid_rowconfigure(0, weight=1)
        self.video_frame.grid_columnconfigure(0, weight=1)

        self.video_label = ctk.CTkLabel(self.video_frame, text="대기 중...\n(카메라를 켜거나 영상을 업로드하세요)", text_color="gray", font=ctk.CTkFont(size=16))
        self.video_label.grid(row=0, column=0)

        # 2. 오른쪽: 컨트롤 패널 및 점수판 프레임
        self.control_frame = ctk.CTkFrame(self, width=320, corner_radius=15)
        self.control_frame.grid(row=0, column=1, padx=(0, 20), pady=20, sticky="nsew")
        self.control_frame.grid_propagate(False)
        

        # 2-1. 버튼 영역
        ctk.CTkLabel(self.control_frame, text="[ 영상 입력 및 추출 ]", font=ctk.CTkFont(size=16, weight="bold")).pack(pady=(20, 10))
        
        self.btn_upload = ctk.CTkButton(self.control_frame, text="📂 모의면접 영상 업로드", command=self.upload_video)
        self.btn_upload.pack(pady=5, padx=20, fill="x")

        self.btn_extract = ctk.CTkButton(self.control_frame, text="✂️ 시각/음성 데이터 추출", fg_color="#2E8B57", hover_color="#236B43", command=self.extract_data)
        self.btn_extract.pack(pady=5, padx=20, fill="x")

        ctk.CTkLabel(self.control_frame, text="[ 실시간 모의면접 ]", font=ctk.CTkFont(size=16, weight="bold")).pack(pady=(30, 10))
        
        self.btn_start_cam = ctk.CTkButton(self.control_frame, text="🎥 웹캠 켜기 / 면접 시작", command=self.start_webcam)
        self.btn_start_cam.pack(pady=5, padx=20, fill="x")

        self.btn_stop_cam = ctk.CTkButton(self.control_frame, text="⏹️ 면접 종료 및 저장", fg_color="#C62828", hover_color="#8E0000", command=self.stop_webcam, state="disabled")
        self.btn_stop_cam.pack(pady=5, padx=20, fill="x")

        # 2-2. 점수 시각화 및 산출 영역
        ctk.CTkLabel(self.control_frame, text="[ AI 면접 분석 결과 ]", font=ctk.CTkFont(size=16, weight="bold")).pack(pady=(40, 15))
        
        self.score_bars = {}
        self.score_text_labels = {}
        metrics = ["시선 처리", "목소리 크기", "답변 명확성", "표정 긍정성"]
        
        for metric in metrics:
            lbl_frame = ctk.CTkFrame(self.control_frame, fg_color="transparent")
            lbl_frame.pack(fill="x", padx=20, pady=(5, 0))
            
            ctk.CTkLabel(lbl_frame, text=metric, font=ctk.CTkFont(size=13)).pack(side="left")
            score_lbl = ctk.CTkLabel(lbl_frame, text="0점", font=ctk.CTkFont(size=13, weight="bold"))
            score_lbl.pack(side="right")
            self.score_text_labels[metric] = score_lbl
            
            progress = ctk.CTkProgressBar(self.control_frame, height=12)
            progress.set(0.0)
            progress.pack(fill="x", padx=20, pady=(0, 10))
            self.score_bars[metric] = progress

        self.btn_get_score = ctk.CTkButton(self.control_frame, text="🎯 AI 면접 점수 산출하기", height=40, font=ctk.CTkFont(size=15, weight="bold"), fg_color="#1E88E5", hover_color="#1565C0", command=self.display_dummy_scores)
        self.btn_get_score.pack(pady=20, padx=20, fill="x")

    # --- 기능 구현 부 ---

    def upload_video(self):
        self.stop_webcam(force_stop=True) # 기존 재생 영상이나 웹캠을 안전하게 종료
        filepath = filedialog.askopenfilename(title="Select Video File", filetypes=(("MP4 files", "*.mp4"), ("AVI files", "*.avi"), ("All files", "*.*")))
        if filepath:
            self.current_video_path = filepath
            self.vid = cv2.VideoCapture(filepath)
            self.update_frame()
            messagebox.showinfo("알림", "영상이 로드되었습니다. 추출 또는 점수 산출을 진행해보세요.")

    def extract_data(self):
        if not self.current_video_path:
            messagebox.showwarning("경고", "업로드된 영상이나 녹화 완료된 영상이 없습니다.")
            return

        if VideoFileClip is None:
            messagebox.showerror("에러", "moviepy 라이브러리가 설치되지 않았습니다.")
            return

        try:
            video = VideoFileClip(self.current_video_path)
            audio_path = "extracted_audio.mp3"
            video.audio.write_audiofile(audio_path, logger=None)
            messagebox.showinfo("성공", f"음성 데이터가 '{audio_path}'로 추출되었습니다.")
        except Exception as e:
            messagebox.showerror("추출 실패", f"에러가 발생했습니다: {e}")

    def start_webcam(self):
        self.stop_webcam(force_stop=True) # ★ 핵심 해결 포인트: 웹캠 켜기 전 기존 화면/루프 완벽 차단
        
        self.vid = cv2.VideoCapture(0)
        
        # 녹화를 위한 VideoWriter 설정
        width = int(self.vid.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(self.vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = 20.0
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        
        self.current_video_path = "recorded_interview.avi"
        self.video_writer = cv2.VideoWriter(self.current_video_path, fourcc, fps, (width, height))
        
        self.is_recording = True
        self.btn_start_cam.configure(state="disabled")
        self.btn_stop_cam.configure(state="normal")
        self.update_frame()

    def stop_webcam(self, force_stop=False):
        self.is_recording = False
        self.btn_start_cam.configure(state="normal")
        self.btn_stop_cam.configure(state="disabled")
        
        # ★ 진행 중인 프레임 업데이트 루프를 안전하게 끊어줌
        if self.loop_id:
            self.after_cancel(self.loop_id)
            self.loop_id = None
            
        if self.vid:
            self.vid.release()
            self.vid = None
            
        if self.video_writer:
            self.video_writer.release()
            self.video_writer = None

        if not force_stop:
            success_msg = f"✅ 녹화 완료!\n\n영상 저장 위치: {self.current_video_path}\n아래 '점수 산출하기' 버튼을 눌러주세요."
            self.video_label.configure(image="", text=success_msg, text_color="#2E8B57")
        else:
            self.video_label.configure(image="", text="대기 중...\n(카메라를 켜거나 영상을 업로드하세요)", text_color="gray")

    def update_frame(self):
        if self.vid and self.vid.isOpened():
            ret, frame = self.vid.read()
            if ret:
                # 1. 실제 영상 녹화
                if self.is_recording and self.video_writer:
                    self.video_writer.write(frame)

                # 2. UI 표시용 리사이징
                display_frame = cv2.resize(frame, (640, 480))
                display_frame = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)
                
                pil_image = Image.fromarray(display_frame)
                ctk_image = ctk.CTkImage(light_image=pil_image, dark_image=pil_image, size=(640, 480))
                
                self.video_label.configure(image=ctk_image, text="")
                
                # ★ 반복 루프를 변수에 저장하여 언제든 취소할 수 있게 함
                self.loop_id = self.after(self.delay, self.update_frame)
            else:
                self.stop_webcam(force_stop=True)
                # 영상 파일이 끝까지 재생된 경우
                self.video_label.configure(text="✅ 영상 재생 완료\n추출 또는 점수 산출을 진행해보세요.", text_color="#1E88E5", image="")
                return

    def display_dummy_scores(self):
        if not self.current_video_path:
            messagebox.showwarning("경고", "먼저 영상을 업로드하거나 면접 녹화를 완료해주세요.")
            return

        dummy_scores = {
            "시선 처리": 85,
            "목소리 크기": 70,
            "답변 명확성": 92,
            "표정 긍정성": 65
        }
        
        for metric, score in dummy_scores.items():
            normalized_score = score / 100.0
            self.score_bars[metric].set(normalized_score)
            self.score_text_labels[metric].configure(text=f"{score}점")
            
        messagebox.showinfo("분석 완료", "분석 모듈로부터 점수를 산출했습니다!")

if __name__ == "__main__":
    app = MockInterviewApp()
    app.mainloop()

c:\Users\lenac\anaconda3\Lib\site-packages\customtkinter\windows\widgets\core_widget_classes\ctk_base_class.py:179: UserWarning: CTkLabel Warning: Given image is not CTkImage but <class 'str'>. Image can not be scaled on HighDPI displays, use CTkImage instead.

  warnings.warn(f"{type(self).__name__} Warning: Given image is not CTkImage but {type(image)}. Image can not be scaled on HighDPI displays, use CTkImage instead.\n")


In [2]:
pip install customtkinter

   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/296.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/296.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/296.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/296.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/296.1 kB ? eta -:--:--
   ---- ----------------------------------- 30.7/296.1 kB 87.5 kB/s eta 0:00:04
   ---- ----------------------------------- 30.7/296.1 kB 87.5 